# Train Tiny Chemical LLM (Character-Level GPT)

This notebook demonstrates how to train a **very small** Language Model (LLM) from scratch using PyTorch.
We will train it on the **SMILES** strings from your dataset to generate new valid chemical structures.

**Model Architecture:** Tiny Transformer (Decoder-only, GPT-style)
**Tokenizer:** Character-level (Simple & Small)
**Data:** `universal_training_dataset_IR.jsonl`

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import pandas as pd
import json
import numpy as np
from pathlib import Path

# Check GPU
device =  'cpu'
print(f"Using device: {device}")

Using device: cpu


## 1. Load Data & Create Tokenizer

In [2]:
# Load SMILES from JSONL
data_path = "../data/for_train/universal_training_dataset_IR.jsonl"
smiles_list = []

try:
    with open(data_path, 'r') as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                if 'smiles' in record and record['smiles']:
                    smiles_list.append(record['smiles'])
except FileNotFoundError:
    print("Data file not found. Using dummy data for demonstration.")
    smiles_list = [
        'C=CC1=CC=CC=C1', 'CC1=CC=CC=C1', 'CCO', 'CCN', 'C(=O)O',
        'C1CCCCC1', 'c1ccccc1', 'CC(=O)C', 'CC(=O)O', 'C1=CC=CC=C1',
        'CN1C=NC2=C1C(=O)N(C(=O)N2C)C', 'CC(C)CC1=CC=C(C=C1)C(C)C(=O)O'
    ] * 10 # Repeat to ensure enough data

text = "\n".join(smiles_list)
print(f"Total SMILES: {len(smiles_list)}")
print(f"Total characters: {len(text)}")

# Character-level Tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocab size: {vocab_size}")
print(f"Vocabulary: {''.join(chars)}")

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Train/Val Split
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

Data file not found. Using dummy data for demonstration.
Total SMILES: 120
Total characters: 1479
Vocab size: 10
Vocabulary: 
()12=CNOc


## 2. Define Tiny GPT Model

In [3]:
# Hyperparameters (Tiny Config)
batch_size = 32
block_size = 64 # Context length
if len(train_data) < block_size:
    block_size = len(train_data) - 1
    print(f"⚠️ Data too small, adjusting block_size to {block_size}")

max_iters = 1000
eval_interval = 100
learning_rate = 1e-3
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * C**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return self.dropout(out)

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = TinyGPT()
m = model.to(device)
print(f"Model parameters: {sum(p.numel() for p in m.parameters())/1e3} K")

Model parameters: 204.682 K


## 3. Training Loop

In [4]:
def get_batch(split):
    data_src = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_src) - block_size, (batch_size,))
    x = torch.stack([data_src[i:i+block_size] for i in ix])
    y = torch.stack([data_src[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Starting training...")
for iter in range(max_iters):
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % eval_interval == 0:
        print(f"step {iter}: loss {loss.item():.4f}")

print(f"Final loss: {loss.item():.4f}")

Starting training...
step 0: loss 2.3400
step 100: loss 0.8801
step 200: loss 0.4686
step 300: loss 0.1641
step 400: loss 0.1101
step 500: loss 0.0765
step 600: loss 0.0600
step 700: loss 0.0728
step 800: loss 0.0612
step 900: loss 0.0525
Final loss: 0.0565


## 4. Generate New SMILES

In [5]:
# Generate from context 'C'
context = torch.zeros((1, 1), dtype=torch.long, device=device)
context[0,0] = stoi['C']

print("Generating SMILES...")
print(decode(m.generate(context, max_new_tokens=100)[0].tolist()))

Generating SMILES...
CC1
c1ccccc1
CC(=O)C
CC(=O)O
C1=CC=CC=C1
CN1C=NC2=C1C(=O)N(C(=O)N2C)C
CCC(C)CC1=CC=C1)CC=C(C=C1)C(C)C
